# RelayVSR trên Kaggle (upscale video bằng model diffusion, không upscale từng frame)

Model [Comfy-Org/RelayVSR](https://huggingface.co/Comfy-Org/RelayVSR) (bản repack cho ComfyUI của
[kopper/RelayVSR](https://huggingface.co/kopper/RelayVSR)) — **video super-resolution** kiểu mới:

* Chỉ **mỗi frame thứ 15** đi qua model Wan 2.2 (sparse keyframes); phần còn lại do decoder "flash"
  tái tạo → nhanh hơn hẳn cách upscale từng frame bằng Real-ESRGAN.
* Hệ số phóng đại do checkpoint quyết định: **x2** (`relayvsr_x2_lightx2v_int8_convrot`) hoặc **x4**.
* Chạy 1 step, CFG 1.0; giữ luôn **audio gốc**; giảm cả hiện tượng nhấp nháy theo thời gian (temporal flicker).
* Chạy trong **ComfyUI core** (node `RelayVSRConditioning` + `VSRDecode`, master sau **06/11/2026**) —
  notebook này điều khiển ComfyUI headless qua HTTP API, không cần mở giao diện.

**Chuẩn bị trên Kaggle:** *Settings → Accelerator = **GPU T4 x2***, *Internet = **On***.
Cần tải ~5,3 GB weight (x2) — chọn x4 thì cũng ~5,3 GB. Thời gian: 1 clip 720p ~10 s có thể mất vài phút
trên T4 (chưa đo được ở máy dev vì không có GPU — xem mục cuối).

In [ ]:
#@title 1) Kiểm tra môi trường
import os, shutil, subprocess, sys
from pathlib import Path

!nvidia-smi | head -12
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| số GPU:", torch.cuda.device_count())
free = shutil.disk_usage("/kaggle/working" if Path("/kaggle/working").exists() else ".")
print(f"đĩa trống: {free.free / 1073741824:.1f} GB | working: {free.total / 1073741824:.1f} GB")
print("python", sys.version.split()[0])

In [ ]:
#@title 2) Cài ComfyUI (bản master — bắt buộc, bản release chưa có node RelayVSR)
REPO_URL = "https://github.com/hoannguyen171/UPSCALEVIDEO.git"  #@param {type:"string"}
WORK = Path("/kaggle/working" if Path("/kaggle/working").exists() else ".")
CUI = WORK / "valo-upscale-gpu"
COMFY = WORK / "ComfyUI"

if not (CUI / ".git").exists():
    !git clone --depth 1 -q "{REPO_URL}" "{CUI}"
else:
    !git -C "{CUI}" pull -q --ff-only

if not (COMFY / "main.py").exists():
    !git clone --depth 1 -q https://github.com/comfyanonymous/ComfyUI.git "{COMFY}"
else:
    !git -C "{COMFY}" pull -q --ff-only
!git -C "{COMFY}" log -1 --date=short --format="ComfyUI %h (%cd)"

# Cài dependency ComfyUI nhưng KHÔNG để pip thay torch CUDA của Kaggle bằng bản CPU
req = (COMFY / "requirements.txt").read_text(encoding="utf-8").splitlines()
skip = ("torch", "torchvision", "torchaudio")
(COMFY / "requirements-notorch.txt").write_text(
    "\n".join(l for l in req if l.split("==")[0].strip() not in skip), encoding="utf-8")
!pip install -q -r "{COMFY}/requirements-notorch.txt"
# torchaudio bản khớp torch đang có (ComfyUI chỉ dùng cho vài node audio)
tv = torch.__version__
base, _, cu = tv.partition("+")
tag = cu if cu.startswith("cu") else "cpu"
try:
    !pip install -q --no-deps --index-url https://download.pytorch.org/whl/{tag} "torchaudio=={base}"
except Exception as exc:
    print("Bỏ qua torchaudio:", exc)
print("torch sau khi cài (phải giữ nguyên bản CUDA):", __import__("torch").__version__)

In [ ]:
#@title 3) Tải weight RelayVSR (có thể dùng HF_TOKEN cho nhanh hơn)
#@markdown Repo `Comfy-Org/RelayVSR` là **công khai**, token chỉ để tăng hạn mức tốc độ tải.
#@markdown Token nên để ở **Kaggle Secrets** (Add-ons → Secrets, tên `HF_TOKEN`) thay vì gõ vào notebook.
HF_TOKEN = ""  #@param {type:"string"}
WEIGHTS_SOURCE = "/kaggle/input"  #@param {type:"string"}
SCALE = 2  #@param [2, 4] {type:"raw"}

import os
sys.path.insert(0, str(CUI))
import importlib, valo_upscale.comfy_upscale as cu
importlib.reload(cu)

if not HF_TOKEN:  # thử lấy từ Kaggle Secrets nếu có
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        print("đã lấy HF_TOKEN từ Kaggle Secrets")
    except Exception:
        HF_TOKEN = ""
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

!pip install -q huggingface_hub
for path in cu.download_models(str(COMFY), int(SCALE), token=HF_TOKEN or None,
                               source_dir=WEIGHTS_SOURCE or None, log=lambda t: print(t, flush=True)):
    print(f"  ✔ {path} ({Path(path).stat().st_size / 1073741824:.2f} GB)")
!ls -la "{COMFY}/models/diffusion_models" "{COMFY}/models/vae"


In [ ]:
#@title 4) Chạy ComfyUI headless + kiểm tra node/weight
LOW_VRAM = False  #@param {type:"boolean"}

import time
extra_args = ("--lowvram",) if LOW_VRAM else ()
proc = cu.start_server(COMFY, port=8188, extra_args=extra_args, log_path=str(WORK / "comfy.log"))
started = time.time()
cu.wait_ready("http://127.0.0.1:8188", timeout=900)
print(f"ComfyUI sẵn sàng sau {time.time() - started:.0f}s")

model_file, vae_file = cu.model_files(int(SCALE))
state = cu.check_ready("http://127.0.0.1:8188", model_file, vae_file)
print("model :", model_file)
print("vae   :", vae_file)
print("sẵn sàng:", state["ok"], "| node thiếu:", state["missing_nodes"], "| vấn đề:", state["problems"])
if not state["ok"]:
    print("Gợi ý: ComfyUI phải là master sau 06/11/2026; weight phải nằm đúng models/diffusion_models và models/vae.")


In [ ]:
#@title 5) Chọn video đầu vào (đường dẫn local, /kaggle/input, hoặc LINK CÔNG KHAI)
INPUT = ""  #@param {type:"string"}
LIMIT = 1  #@param {type:"integer"}
OUT_DIR = str(WORK / "relayvsr_out")  #@param {type:"string"}

import glob
from valo_upscale import remote

videos = []
if not INPUT.strip():
    videos = sorted(glob.glob("/kaggle/input/**/*.mp4", recursive=True))[:LIMIT]
    print("Tự quét /kaggle/input:", videos)
elif INPUT.strip().startswith(("http://", "https://")):
    items = remote.list_remote(INPUT.strip())
    videos = [str(p) for p in remote.download_remote(items[:LIMIT], str(WORK / "relayvsr_in"),
                                                      progress=lambda t: print(t, flush=True))]
    print("Đã tải từ link:", videos)
else:
    videos = [INPUT.strip()]
print("Sẽ xử lý:", videos)

In [ ]:
#@title 6) Upscale (RelayVSR) — chạy và xem kết quả
DETAIL = 1.0  #@param {type:"slider", min:0.0, max:1.0, step:0.05}
STEPS = 1  #@param [1] {type:"raw"}
CFG = 1.0  #@param {type:"number"}
SEED = 0  #@param {type:"integer"}
TIMEOUT = 7200  #@param {type:"integer"}

from pathlib import Path
from IPython.display import Video, display

results = []
for index, video in enumerate(videos, start=1):
    out = Path(OUT_DIR) / f"{Path(video).stem}_relayvsr_x{SCALE}.mp4"
    print(f"[{index}/{len(videos)}] {Path(video).name} → {out.name}")
    result = cu.upscale_video(
        video, out,
        comfy_dir=str(COMFY), base="http://127.0.0.1:8188", scale=int(SCALE),
        steps=int(STEPS), cfg=float(CFG), detail=float(DETAIL), seed=int(SEED), timeout=int(TIMEOUT),
        progress=lambda sec, state: print(f"   [{sec}s] {state}", flush=True),
    )
    results.append(result)
    print(f"   ✔ {result['info']['width']}x{result['info']['height']} | "
          f"{result['info']['nb_frames']} frame | audio {result['info']['has_audio']} | {result['elapsed']:.1f}s")
    print(f"   kích thước đúng như mong đợi ({result['expected_size']}): {result['size_ok']}")
    display(Video(result["output_path"], embed=True, width=560))

## 7) Ghi chú & mẹo

| Việc | Cách làm |
|---|---|
| Chọn x2 hay x4 | x2 (native 2×, nhanh hơn) cho clip AI 720p→1440p; x4 cho 1080p→4K. Hệ số do checkpoint, không phải tham số. |
| Muốn nét hơn / bớt nét | `DETAIL` 0→1 (1 = giữ toàn bộ chi tiết model thêm vào bản bicubic). |
| Hết VRAM (T4 16 GB) | Bật `LOW_VRAM` ở ô 4, hoặc chia clip thành đoạn ngắn. |
| Không muốn tải lại 5,3 GB mỗi phiên | Sau lần chạy đầu: **Save version** → tab **Output** → *New Dataset* từ thư mục `ComfyUI/models`. Phiên sau *Add Input* dataset đó rồi để `WEIGHTS_SOURCE="/kaggle/input"` — notebook **symlink** sang, không tốn thêm 5,3 GB đĩa. |
| Tải chậm / bị giới hạn hạn mức | Điền `HF_TOKEN` ở ô 3 (repo `Comfy-Org/RelayVSR` **công khai**, token chỉ để tăng tốc); nên lưu token ở **Kaggle Secrets** (Add-ons → Secrets, tên `HF_TOKEN`) thay vì gõ vào notebook. |
| ComfyUI báo thiếu node | ComfyUI phải là **master sau 06/11/2026** (PR #17511) — chạy lại ô 2 để cập nhật. |
| Lỗi khi nạp model int8 (kiến trúc Turing) | Đổi sang weight fp của `kopper/RelayVSR` (`wan.safetensors` 9,6 GB + `flash.safetensors` 122 MB) và trỏ `model_file`/`vae_file` sang tên file đó. |
| Muốn dùng cả 2 GPU | Đường LightX2V có config `relayvsr_x2_sp.json` (sequence parallel 2 GPU) — cài LightX2V thay vì ComfyUI. |
| Kết quả ở đâu | `/kaggle/working/relayvsr_out` — bấm **Save version** rồi tải ở tab **Output**. |

**Chưa kiểm chứng được ở máy dev:** phần suy luận RelayVSR cần GPU (máy dev không có NVIDIA), nên chỉ kiểm
được: node/input của ComfyUI đúng, workflow hợp lệ, client + các node video core chạy thật (graph stub ra mp4
giữ audio), tải weight/đường dẫn/symlink. Lần chạy đầu trên Kaggle nên thử **1 clip ngắn** để đo thời gian
trước khi chạy clip dài.

In [ ]:
#@title 8) Liệt kê / nén kết quả
import shutil
from pathlib import Path
from valo_upscale import probe

out_dir = Path(OUT_DIR)
files = sorted(out_dir.glob("*.mp4")) if out_dir.exists() else []
print(f"{out_dir}: {len(files)} file")
for f in files:
    info = probe.get_media_info(f)
    print(f"  • {f.name} — {f.stat().st_size / 1048576:.1f} MB | "
          f"{info['width']}x{info['height']} | audio {info['has_audio']}")
if files:
    zip_path = shutil.make_archive(str(out_dir.parent / "relayvsr_ket_qua"), "zip", root_dir=str(out_dir))
    print("ZIP:", zip_path)
    if not Path("/kaggle/working").exists():
        from google.colab import files as colab_files
        colab_files.download(zip_path)
